# Batch Processing Demo

This notebook demonstrates batch processing capabilities:

1. **Batch Sequence Search** - Processing multiple motif libraries
2. **Batch Structure Search** - Processing multiple structure files
3. **Multi-Organism Analysis** - Comparing motifs across species
4. **Parallel Processing** - Tips for large-scale analysis

---
## Setup

In [ ]:
import sys
import os
import pandas as pd
import json
from glob import glob
from datetime import datetime

sys.path.insert(0, os.path.abspath('../sequence_motif'))
sys.path.insert(0, os.path.abspath('../structure_motif'))


from file_converter import process_protein_files
from motif_searcher import run_motif_search
from uniprot_api import search_uniprot, to_csv
from search_3d_motif import search_single_file, parse_motif_file

PROTEIN_FILES_DIR = '../protein_files'
MOTIF_LIBRARIES_DIR = '../sequence_motif/motif_libraries'
STRUCTURE_MOTIFS_DIR = '../structure_motif/motifs'
OUTPUT_DIR = '../outputs/batch_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup complete!')

---
## 1. Batch Sequence Search: Multiple Motif Libraries

Search sequences against multiple motif libraries in one run.

In [ ]:
# List available motif libraries
print('Available motif libraries:')
motif_libs = [f for f in os.listdir(MOTIF_LIBRARIES_DIR) if f.endswith('.csv')]
for lib in motif_libs:
    path = os.path.join(MOTIF_LIBRARIES_DIR, lib)
    try:
        df = pd.read_csv(path)
        print(f'  {lib}: {len(df)} motifs')
    except:
        print(f'  {lib}: (error reading)')

In [ ]:
# Prepare sequences
sequences_csv = os.path.join(OUTPUT_DIR, 'batch_sequences.csv')
process_protein_files(PROTEIN_FILES_DIR, sequences_csv)
print(f'Prepared sequences for batch processing')

In [ ]:
# Batch search across multiple motif libraries
original_dir = os.getcwd()
os.chdir('../sequence_motif')

batch_results = {}
##some example libraries to search through
libraries_to_search = ['kinase_substrate_motifs.csv', 'protease_sample_motifs.csv']

print('Running batch search across motif libraries...')
print('='*50)

for lib in libraries_to_search:
    lib_path = os.path.join(MOTIF_LIBRARIES_DIR, lib)
    if not os.path.exists(lib_path):
        print(f'  Skipping {lib} (not found)')
        continue
    
    output_file = os.path.join(OUTPUT_DIR, f'{lib.replace(".csv", "")}_results.csv')
    print(f'\nSearching with {lib}...')
    
    # Detect columns
    lib_df = pd.read_csv(lib_path)
    motif_col = [c for c in lib_df.columns if 'motif' in c.lower() and 'name' not in c.lower()][0]
    name_col = [c for c in lib_df.columns if 'name' in c.lower()][0] if any('name' in c.lower() for c in lib_df.columns) else motif_col
    
    ##command for the search
    run_motif_search(
        motifs_file=lib_path,
        motif_column=motif_col,
        motif_name_column=name_col,
        sequences_file=sequences_csv,
        sequence_column='sequence',
        output_file=output_file,
        name_column='name'
    )
    batch_results[lib] = output_file
    print(f'  Results saved to {output_file}')

os.chdir(original_dir)
print('\nBatch sequence search complete!')

---
## 2. Batch Structure Search: Multiple Motif Definitions

Search structures against multiple structural motif definitions.

In [ ]:
# List available structure motif definitions
print('Available structure motif definitions:')
struct_motifs = [f for f in os.listdir(STRUCTURE_MOTIFS_DIR) if f.endswith('.json')]
for m in struct_motifs:
    path = os.path.join(STRUCTURE_MOTIFS_DIR, m)
    with open(path) as f:
        mdef = json.load(f)
    print(f'  {m}: {mdef.get("motif_name", "Unknown")}')

In [ ]:
# Get all structure files
structure_files = sorted(glob(os.path.join(PROTEIN_FILES_DIR, '*.pdb')) + glob(os.path.join(PROTEIN_FILES_DIR, '*.cif')))
print(f'Found {len(structure_files)} structure files')

In [ ]:
# Batch search: all structures against all motif definitions
print('Running batch structure search...')
print('='*50)

all_struct_results = []

for motif_file in struct_motifs:
    motif_path = os.path.join(STRUCTURE_MOTIFS_DIR, motif_file)
    with open(motif_path) as f:
        motif_def = json.load(f)
    
    motif_name = motif_def.get('motif_name', motif_file)
    print(f'\nSearching for: {motif_name}')
    
    # Create output directory for this motif
    motif_output_dir = os.path.join(OUTPUT_DIR, motif_file.replace('.json', ''))
    os.makedirs(motif_output_dir, exist_ok=True)
    
    motif_results = []
    for sf in structure_files:
        found = search_single_file(sf, motif_def)
        result = {
            'motif': motif_name,
            'structure': os.path.basename(sf),
            'count': len(found),
            'matches': found
        }
        motif_results.append(result)
        
        # Save individual result
        out_json = os.path.join(motif_output_dir, os.path.basename(sf).rsplit('.', 1)[0] + '.json')
        with open(out_json, 'w') as f:
            json.dump(result, f, indent=2)
    
    all_struct_results.extend(motif_results)
    found_count = sum(1 for r in motif_results if r['count'] > 0)
    print(f'  Found in {found_count}/{len(structure_files)} structures')

print('\nBatch structure search complete!')

In [ ]:
# Create summary DataFrame
summary_df = pd.DataFrame([{'motif': r['motif'], 'structure': r['structure'], 'count': r['count']} for r in all_struct_results])
pivot = summary_df.pivot_table(index='structure', columns='motif', values='count', fill_value=0)
print('Structure x Motif Summary:')
pivot

---
## 3. Multi-Organism Comparison

Compare motif occurrences across different organisms.

In [ ]:
# Define organisms to compare
organisms = [
    ('Human', 'Homo sapiens'),
    ('Mouse', 'Mus musculus'),
    ('Zebrafish', 'Danio rerio')
]

print('Fetching sequences from multiple organisms...')
organism_sequences = {}

In [ ]:
# Fetch sequences for each organism using the uniprot API
for name, sci_name in organisms:
    print(f'\nFetching {name} kinases...')
    query = f'(organism_name:"{sci_name}") AND (protein_name:"kinase")'
    try:
        data = search_uniprot(query, limit=10) ##limit of 10 each
        csv_path = os.path.join(OUTPUT_DIR, f'{name.lower()}_kinases.csv')
        to_csv(data, csv_path)
        organism_sequences[name] = csv_path
        print(f'  Retrieved sequences')
    except Exception as e:
        print(f'  Error: {e}')

In [ ]:
# Search each organism's sequences for the same motifs
comparison_motifs = pd.DataFrame({
    'motif_name': ['PKA_Site', 'PKC_Site', 'ATP_Binding'],
    'motifs': ['RRx[ST]', '[ST]x[RK]', 'GxGxx[GS]']
})
comparison_file = os.path.join(OUTPUT_DIR, 'comparison_motifs.csv')
comparison_motifs.to_csv(comparison_file, index=False)

print('Running comparative analysis...')
original_dir = os.getcwd()
os.chdir('../sequence_motif')

comparison_results = {}
for name, seq_path in organism_sequences.items(): ##this time going through for each organism and sequence
    if not os.path.exists(seq_path):
        continue
    
    # Read to find sequence column
    df = pd.read_csv(seq_path)
    seq_col = 'Sequence' if 'Sequence' in df.columns else 'sequence'
    name_col = 'Entry' if 'Entry' in df.columns else 'name'
    
    output = os.path.join(OUTPUT_DIR, f'{name.lower()}_motif_results.csv')
    print(f'  Searching {name}...')
    
    run_motif_search(
        motifs_file=comparison_file,
        motif_column='motifs',
        motif_name_column='motif_name',
        sequences_file=seq_path,
        sequence_column=seq_col,
        output_file=output,
        name_column=name_col
    )
    comparison_results[name] = output

os.chdir(original_dir)
print('Comparative analysis complete!')

---
## 4. Tips for Large-Scale Batch Processing

In [ ]:
# Example: Process in chunks for very large datasets
def process_in_chunks(sequence_file, chunk_size=1000):
    """Process a large sequence file in chunks."""
    df = pd.read_csv(sequence_file)
    total_chunks = (len(df) + chunk_size - 1) // chunk_size
    
    print(f'Processing {len(df)} sequences in {total_chunks} chunks')
    
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        chunk_file = f'temp_chunk_{i//chunk_size}.csv'
        chunk.to_csv(chunk_file, index=False)
        # Process chunk here...
        print(f'  Chunk {i//chunk_size + 1}/{total_chunks}: {len(chunk)} sequences')
        os.remove(chunk_file)
    
    return total_chunks

# Demonstration (not actually run on large data)
print('Chunk processing ready for large datasets')

In [ ]:
# Example: Command line batch processing script
batch_script = '''
#!/bin/bash
# Batch processing example script

# Process multiple organisms
for organism in "human" "mouse" "zebrafish"; do
    echo "Processing $organism..."
    python main.py uniprot --organism "$organism" --output_csv ${organism}_seqs.csv --limit 100
    python main.py search \\
        --motifs motif_libraries/kinase_substrate_motifs.csv \\
        --motif_column motifs \\
        --sequences ${organism}_seqs.csv \\
        --sequence_column sequence \\
        --output ${organism}_results.csv
done

# Process multiple motif libraries
for lib in kinase protease glycosylation; do
    echo "Searching with ${lib} motifs..."
    python main.py search \\
        --motifs motif_libraries/${lib}_motifs.csv \\
        --motif_column motifs \\
        --sequences all_sequences.csv \\
        --output ${lib}_results.csv
done

# Structure batch processing
for motif in catalytic_triad kinase_active_site; do
    python structure_motif/search_3d_motif.py \\
        -i protein_files \\
        -m structure_motif/motifs/${motif}.json \\
        -o outputs/${motif} \\
        -s ${motif}_summary.csv
done
'''

script_path = os.path.join(OUTPUT_DIR, 'batch_process.sh')
with open(script_path, 'w') as f:
    f.write(batch_script)
print(f'Batch script saved to: {script_path}')
print('\nScript contents:')
print(batch_script)